# Introduction
- Date: 20241218
- Last Update: 20250501
- Author: BL
- This is a script to make polished figured panels for the spatial lung manuscript

Update log:
- Swapping to newest dataset
- Updating to match the newest panels

In [1]:
import pickle
import os
import itertools
import scipy
import csv
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import gseapy as gp
import anndata as ad
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from matplotlib.pyplot import rc_context
from matplotlib.colors import LogNorm, Normalize
from scipy import sparse, stats
from copy import copy
from scipy.stats import norm
from sklearn.neighbors import KernelDensity
from typing import Tuple
from matplotlib.colors import LinearSegmentedColormap, Normalize

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
sc.settings.verbosity = 0

In [3]:
output_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/panels/Fig2/'
sup_loc = '/Genomics/chanlab/blaw/Spatial_Lung/sandbox/panels/FigS2/'

In [4]:
os.chdir(output_loc)

# Load helpful functions and objects

In [5]:
all_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science', 'E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
dbit_datasets = ['E11lungR1', 'E11lungR2', 'E11lungR3', 'E11lungR4', 'E11lungR5', 'E12lungR2', 'E12lungR3', 'E12lungR4', 'E12lungR5', 'E13lungR1', 'E13lungR2', 'E13lungR3', 'E13lungR4']
ref_datasets = ['E11_iScience_R1', 'E12_DevCell', 'E12_CellReports_R1', 'E13_CellReports', 'E12_Science']
timepoints = ['E11.5', 'E12.5', 'E13.5']
cellstates = ['MesenchymeProgenitor', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'Precartilage', 'Pericyte',
              'Mesothelium', 'Epithelial(Distal)', 'Epithelial(Proximal)', 'VascularEndothelial',
              'Immune', 'Blood', 'Neuron', 'Heart']
epi_subcluster_cellstates = ['Distal', 'DistalIntermediate', 'ProximalIntermediate', 'Proximal']
mes_subcluster_cellstates = ['MP1', 'MP2', 'MP3', 'SmoothMuscleProgenitor', 'SmoothMuscle', 'MatrixMesenchyme', 'MatrixMesenchymeProgenitor', 'Precartilage', 'Pericyte']

# figure out color scheme
clusterColors = {'Neuron': 'limeGreen',
                 'MatrixMesenchyme': 'DarkRed',
                 'MesenchymeProgenitor': 'RoyalBlue',
                 'Epithelial(Distal)': 'SkyBlue',
                 'Epithelial(Proximal)': 'Red',
                 'DbitEpithelialBridge': 'Cyan',
                 'SmoothMuscleProgenitor': 'Magenta',
                 'SmoothMuscle': 'Purple',
                 'VascularEndothelial': 'gold',
                 'Mesothelium': 'darkgreen',
                 'Precartilage': 'darkgoldenrod',
                 'Immune': 'DarkOrange',
                 'Heart': 'Gray',
                 'Blood': 'tan',
                 'Pericyte': 'lightpink'}

epiSubclusterColors = {'Proximal': 'Red',
                    'ProximalIntermediate':'darkred',
                    'DistalIntermediate':'darkblue',
                    'Distal':'SkyBlue'}

mesSubclusterColors = {'MP1': 'RoyalBlue',
                       'MP2':'lawngreen',
                       'MP3':'Orange',
                       'MP4':'salmon',
                       'MatrixMesenchyme':'DarkRed',
                       'MatrixMesenchymeProgenitor': 'Red',
                       'SmoothMuscleProgenitor': 'Magenta',
                       'SmoothMuscle':'Purple',
                       'Pericyte': 'lightpink',
                       'Precartilage':'darkgoldenrod'}

phaseDict = {'G1': 'Blue', 'G2M': 'Orange', 'S': 'Green'}

greens = copy(mpl.cm.Greens)
greens.set_under("lightgray")
reds = copy(mpl.cm.Reds(np.linspace(0,1,40)))
reds = mpl.colors.ListedColormap(reds[5:])
reds.set_under("lightgray")

# Add silver as the lower color for the expression colormap
colors = ["silver", "red"]  # Colors for the gradient
expr_cmap = LinearSegmentedColormap.from_list("silver_to_red", colors)
expr_cmap.set_over("red") 
norm = Normalize(vmin=0, vmax=10)

In [6]:
def plot_spatial_groups(adata, 
                        meta_col,
                        background_adata = False, 
                        palette = False, 
                        show = True, 
                        output_loc = False, 
                        name = 'spatial_plot', 
                        size = 20):
    '''
    input
        - adata: An anndata object. Assumes 14 dbit datasets (split by 'orig.ident'), labeled as dbit in the 'tech' column. Assumes there is an x_loc and y_loc col
        - background_adata: A second anndata object that contains all the cells to plot in the background (if adata is a subset).
            if not provided then it will take adata as the background adata.
        - meta_col: A column in the .obs table or gene in the .var table
        - palette: A dictionary that contains a desired color for each value in the meta_col column or a cmap for a quantitative meta_col.
        - show: True or False on whether to show the plot. Default is False
        - output_loc: Whether or not to save the plot and what folder location. Default is False. 
        - name: The name to give the output pdf if saving the plot
        - size: The dot size for the scatter plot
    returns
        creates spatial scatter plots for the 14 lung spatial datasets. Options to save and show the plot
    '''
    # check if the output_loc folder exists, return an error if not
    if output_loc != False:
        if not os.path.exists(output_loc):
            print('Error: output_loc folder does not exist')
            return
        
    # check if meta_col exists, returns an error if not
    if meta_col not in adata.obs.columns and meta_col not in adata.var.index:
        print('Error: meta_col not in .obs or .var tables')
        return
    
    if type(background_adata) == type(False):
        background_adata = adata
    
    # Make a temp object to work with
    temp_adata = adata.copy()
    
    # Add the gene expr values to the obs table if plotting a gene
    if meta_col in adata.var.index:
        temp_adata.obs[meta_col] = temp_adata[:, meta_col].X.todense()
    
    timepoints = ['E11.5', 'E12.5', 'E13.5']
    fig, axs = plt.subplots(3, 5, figsize = (35, 21))
    row_count = 0
    for time in timepoints:
        datasets = temp_adata.obs[(temp_adata.obs['timepoint'] == time) & (temp_adata.obs['tech'] == 'DBiT_seq')]['orig.ident'].unique()

        col_count = 0
        for exp in datasets:
            temp_ax = axs[row_count, col_count]
            col_count += 1
            
            # Plot full structure in gray from the background adata
            all_cell_locs = background_adata.obs[(background_adata.obs['orig.ident'] == exp)].index
            x = []
            y = []

            for i in all_cell_locs:
                y.append(background_adata.obs.loc[i, 'y_loc'])
                x.append(background_adata.obs.loc[i, 'x_loc'])
            # Use silver with a default alpha to match the lower bound of the expression plots
            temp_ax.scatter(x, y, color = 'silver', s = size)
        
            # plot the quantitative variable
            temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp)].index
            
            if type(temp_adata.obs[meta_col][0]) != str:
                x = [0]
                y = [0]
                z = [temp_adata.obs[temp_adata.obs['tech'] == 'DBiT_seq'][meta_col].max()]
                for i in temp_locs:
                    z.append(temp_adata.obs.loc[i, meta_col])
                    y.append(temp_adata.obs.loc[i, 'y_loc'])
                    x.append(temp_adata.obs.loc[i, 'x_loc'])
                if palette != False:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = palette)
                else:
                    colorbar = temp_ax.scatter(x, y, c = z, s = size, cmap = 'viridis')
            else:
                labels = ['background']
                state_list = temp_adata.obs[meta_col].unique().tolist()
                for state in state_list:
                    temp_locs = temp_adata.obs[(temp_adata.obs['orig.ident'] == exp) & (temp_adata.obs[meta_col] == state)].index

                    x = [0]
                    y = [0]
                    s = [size]
                    for i in temp_locs:
                        y.append(temp_adata.obs.loc[i, 'y_loc'])
                        x.append(temp_adata.obs.loc[i, 'x_loc'])
                        s.append(size)
                    labels.append(state)

                    if palette != False:
                        temp_ax.scatter(x, y, s = size, color = palette[state])
                    else:
                        temp_ax.scatter(x, y, s = size)
                    if row_count == 0 and col_count == 5:
                        temp_ax.legend(labels = labels, bbox_to_anchor = (1.02, 1))
            temp_ax.set_title('{}_{}'.format(exp, meta_col))
            temp_ax.set_xlabel('X Pos')
            temp_ax.set_ylabel('Y Pos')
            temp_ax.set_xlim(0, 52)
            temp_ax.set_ylim(0, 52)

        row_count += 1
    if type(temp_adata.obs[meta_col][0]) != str:
        fig.colorbar(colorbar, ax = axs[2, 4])
    if output_loc != False:
        plt.savefig(output_loc + name, dpi = 300)
    if show:
        plt.show()
    plt.close()

# Load the objects

In [7]:
# Read in the data
lung_adata = sc.read_h5ad('/Genomics/chanlab/blaw/Spatial_Lung/raw_data/pipeline_obj/Lung_dbit_srf_CellReports_Science_with_pseudotime.h5ad')

# Figure 2b
- a UMAP plot colored by cell state

In [32]:
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(lung_adata, color = ['cellstate'], palette = clusterColors, show = False)
    plt.savefig(output_loc + 'Fig.2b_lung_cellstates.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

# Figure 2c
- Spatial map showing the localization of all cellstates

In [ ]:
# Plot the spatial localization of all cellstates across all replicates
plot_spatial_groups(adata = lung_adata,
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.2d_spatial_localization_cellstates.pdf',
                    palette = clusterColors,
                    show = False)

# Figure 2d
- Spatial maps showing the localization of all cellstates, some specific sub states, and expression markers
- Markers include: ['Epcam', 'Actg2', 'Sox9', 'Wt1']

In [72]:
# Plot the spatial localization of just epithelial cells across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['Epithelial(Distal)', 'Epithelial(Proximal)'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.2d_spatial_localization_epithelial.pdf',
                    palette = clusterColors,
                    show = False)

In [73]:
# Plot the spatial localization of just Smooth Muscle cells across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['SmoothMuscleProgenitor', 'SmoothMuscle'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.2d_spatial_localization_smooth_muscle.pdf',
                    palette = clusterColors,
                    show = False)

In [74]:
# Plot the spatial localization of just mesothelium cells across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['Precartilage'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.2d_spatial_localization_precartilage.pdf',
                    palette = clusterColors,
                    show = False)

In [75]:
# Plot the spatial localization of just mesothelium cells across all replicates
plot_spatial_groups(adata = lung_adata[lung_adata.obs['cellstate'].isin(['Mesothelium'])],
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = output_loc,
                    name = 'Fig.2d_spatial_localization_mesothelium.pdf',
                    palette = clusterColors,
                    show = False)

In [38]:
# Plot the spatial localization of each gene's expression across all replicates
genes = ['Epcam', 'Actg2', 'Sox9', 'Wt1']

for gene in genes:
    plot_spatial_groups(adata = lung_adata,
                        meta_col = gene,
                        background_adata = lung_adata,
                        output_loc = output_loc,
                        name = 'Fig.2d_spatial_localization_{}.pdf'.format(gene),
                        palette = expr_cmap,
                        show = False)

# Extended Data
- A UMAP plot colored by technology
- Composition plots per timepoint of the scRNA datasets
- Composition plots per timepoint of the DBiT datasets
- Violin plots of UMI capture of the scRNA datasets
- Violin plots of UMI capture of the DBiT datasets
- Violin plots of gene capture of the scRNA datasets
- Violin plots of gene capture of the DBiT datasets
- UMAPs colored by primary marker genes: ['Epcam', 'Wnt7b', 'Wnt2', 'Actg2', 'Sox9', 'Wt1', 'Aldh1a2', 'Cdh5', 'Sox10', 'Mylk', 'Pdgfrb', 'Prdm16', 'Sox2', 'Tnni1', 'Fcer1g']
- Spatial maps of cellstates across all replicates

In [9]:
os.chdir(sup_loc)

In [11]:
with rc_context({'figure.figsize': (5, 5)}):
    sc.pl.umap(lung_adata, color = ['tech'], show = False)
    plt.savefig(sup_loc + 'Fig.S2_UMAP_tech.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

In [40]:
# Plot the composition per timepoint
time_composition = pd.DataFrame(0, columns = cellstates, index = ['E11.5', 'E12.5', 'E13.5'])
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition.loc[i, j] = len(lung_adata.obs[(lung_adata.obs['timepoint'] == i) & (lung_adata.obs['cellstate'] == j) & (lung_adata.obs['tech'] == 'scRNA_seq')])
        
time_composition_frac = time_composition.copy()
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition_frac.loc[i, j] = time_composition.loc[i, j] / time_composition.loc[i].sum()

fig, ax = plt.subplots(figsize = (5, 5))
# Plot the composition of TLS1 in GFP+ and GFP- groups
time_composition_frac.plot(kind='bar', use_index = True, stacked=True, grid = False, color = clusterColors, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Composition per Timepoint scRNA')
plt.ylabel('% cells')
plt.tight_layout()
plt.savefig(sup_loc + 'Fig.S2_composition_per_timepoint_scRNA.pdf', dpi = 300)
#plt.show()
plt.close()

In [41]:
# Plot the composition per timepoint
time_composition = pd.DataFrame(0, columns = cellstates, index = ['E11.5', 'E12.5', 'E13.5'])
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition.loc[i, j] = len(lung_adata.obs[(lung_adata.obs['timepoint'] == i) & (lung_adata.obs['cellstate'] == j) & (lung_adata.obs['tech'] == 'DBiT_seq')])
        
time_composition_frac = time_composition.copy()
for i in time_composition.index:
    for j in time_composition.columns:
        time_composition_frac.loc[i, j] = time_composition.loc[i, j] / time_composition.loc[i].sum()

fig, ax = plt.subplots(figsize = (5, 5))
# Plot the composition of TLS1 in GFP+ and GFP- groups
time_composition_frac.plot(kind='bar', use_index = True, stacked=True, grid = False, color = clusterColors, ax = ax)
plt.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.title('Composition per Timepoint DBiT')
plt.ylabel('% cells')
plt.tight_layout()
plt.savefig(sup_loc + 'Fig.S2_composition_per_timepoint_DBiT.pdf', dpi = 300)
#plt.show()
plt.close()

In [42]:
# Plot the UMI and Gene capture per DBiT dataset
with rc_context({'figure.figsize': (10, 5)}):
    sc.pl.violin(lung_adata[lung_adata.obs['tech'] == 'DBiT_seq'], ['nCount_RNA', 'nFeature_RNA'], 
                 groupby = 'orig.ident', jitter=0.4, multi_panel= True, rotation = 90, show = False)
    plt.savefig(sup_loc + 'Fig.S2_DBiT_umi_and_gene_capture.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

In [43]:
# Plot the UMI and Gene capture per scRNA dataset
with rc_context({'figure.figsize': (10, 5)}):
    sc.pl.violin(lung_adata[lung_adata.obs['tech'] == 'scRNA_seq'], ['nCount_RNA', 'nFeature_RNA'], 
                 groupby = 'orig.ident', jitter=0.4, multi_panel= True, rotation = 90, show = False)
    plt.savefig(sup_loc + 'Fig.S2_scRNA_umi_and_gene_capture.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

In [13]:
# Plot the expression of marker genes on the UMAP
marker_genes = ['Epcam', 'Wnt7b', 'Sox9', 'Sox2', 'Wnt2', 'Wt1', 'Aldh1a2', 'Pdgfrb', 'Prdm16', 'Ptch1', 'Mylk', 'Actg2', 'Cdh5', 'Sox10', 'Tnni1', 'Fcer1g']

with rc_context({'figure.figsize': (5, 5)}):
    sc.set_figure_params(vector_friendly=True, dpi_save=300) 
    sc.pl.umap(lung_adata, color = marker_genes, cmap = expr_cmap, show = False)
    plt.savefig(sup_loc + 'Fig.S2_marker_gene_umaps.pdf', bbox_inches = "tight", dpi = 300)
    plt.close()

In [45]:
# Plot a violin plot of the scRNA cells per cellstate for each marker gene
for gene in marker_genes:
    with rc_context({'figure.figsize': (6, 2)}):
        sc.pl.violin(lung_adata[lung_adata.obs['tech'] == 'scRNA_seq'], [gene], order = cellstates,
                     groupby = 'cellstate', jitter=0.4, rotation = 90, show = False, palette = clusterColors)
        plt.savefig(sup_loc + 'Fig.S2_scRNA_violin_{}.pdf'.format(gene), bbox_inches = "tight", dpi = 300)
        plt.close()

In [76]:
# Plot the spatial localization of all cellstates across all replicates
plot_spatial_groups(adata = lung_adata,
                    meta_col = 'cellstate',
                    background_adata = lung_adata,
                    output_loc = sup_loc,
                    name = 'Fig.S2_spatial_localization_cellstates.pdf',
                    palette = clusterColors,
                    show = False)